# MANTICORE V7.3.1 — ISOLATED L4 DIAGNOSTIC & TIME BENCHMARK

**Standalone research notebook. Does not run or modify the production V7.3.1 notebook.**

- Reuses the **exact Full model classes and loss/stage functions** extracted from V7.3.1 (embedded source snapshot; SHA256 provenance below). No reduced architecture.
- Runs 8 pooled stages and 8 BTC-only fine-tuning stages, with bounded synthetic shape-compatible batches, checks finite losses / gradients, captures GPU memory and time.
- Performs optional **read-only data archive audit** against mounted Google Drive, including basic contents check. This is not a complete 5-year integrity validation.
- Uses an isolated output directory, never production checkpoint locations.
- Benchmarks do **not** verify alpha, training convergence, actual data correctness, or future return. Time estimates are extrapolations from sampled batches and will be marked approximate.
- This notebook does not download 6 years of data or silently overwrite any production file.


In [ ]:
# 1. Environment and immutable source provenance
import os,sys,json,math,time,hashlib,random,zipfile,re,platform,gc,glob,statistics
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from datetime import datetime,timezone
SOURCE_NOTEBOOK_SHA256 = "e208cbba76f5368734df9ae40ee8b00660619cb672743ffb1c0a0abe194e354b"
LAB_DIR=Path('/content/manticore_v731_isolated_lab') if Path('/content').exists() else Path.cwd()/'manticore_v731_isolated_lab'
LAB_DIR.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16=(DEVICE.type=='cuda' and torch.cuda.is_bf16_supported())
def amp_context():
    return torch.autocast(device_type='cuda',dtype=torch.bfloat16) if USE_BF16 else nullcontext()
random.seed(47);np.random.seed(47);torch.manual_seed(47)
print({'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,'torch':torch.__version__,'bf16':USE_BF16,'source_sha256':SOURCE_NOTEBOOK_SHA256,'lab_dir':str(LAB_DIR)})


In [ ]:
# 2. Production configuration snapshot (separate in-memory CFG; no production writes)
# Config — set DATA_DIR to your mounted Google Drive Parquet folder.
CFG={
 'data_dir':'/content/manticore_v730_crypto_m5',
 'data_glob':None,'use_google_drive':True,'assets':['BTCUSD','ETHUSDT','SOLUSDT'],
 'pool_assets':['BTCUSD','ETHUSDT','SOLUSDT'],'target_asset':'BTCUSD','context_assets':['NSXUSD'],
 'cache_dir':'/content/manticore_v730_pool_finetune_cache','output_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/manticore_v70_trend_specialist_v2/v730_pooled_crypto_btc_finetune_checkpoints',
 'timestamp_semantics':'open','bar_minutes':5,'horizons':[1,3,12,48],'lookback':128,'purge_bars':192,
  'binance_market':'futures/cm','binance_symbol':'BTCUSD_PERP',
  'binance_contract_usd':100.0,'binance_auto_download':True,
 'binance_aux_market':'futures/um','binance_aux_symbols':{'ETHUSDT':'ETHUSDT','SOLUSDT':'SOLUSDT'},
 'binance_aux_archive_base':'/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/um_pooled_aggTrades',
 'binance_aux_m5_base':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_um_pooled_aggtrades',
 'histdata_nsx_zip_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/histdata_tick/NSXUSD',
 'histdata_nsx_m5_cache':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/histdata_nsxusd_tick/NSXUSD_M5.parquet',
 'dollar_bar_target_usd':100000.0,
  'binance_start':'2020-10','binance_end':None,
  'binance_archive_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/cm_BTCUSD_PERP_aggTrades',
  'tick_bar_cache':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_cm_btcusd_aggtrades/BTCUSD_M5.parquet',
 'train_years':5,'val_months':6,'cal_months':3,'sealed_months':3,
 'recent_half_life_days':270.,'recent_uniform_floor':0.15,
 'stress_boost':2.0,'stress_vol_quantile':0.995,'stress_cap':3.0,
 'batch_size':64,'epochs':110,'patience':5,'lr':3e-4,
 'finetune_lr_multiplier':0.15,
 'finetune_epochs':{'A_SSL':3,'B_REGIME':3,'C_GRAPH':3,'D_FORECAST':4,'E_PATH':4,'F_EXPERTS':4,'G_DECISION':3,'H_JOINT':6},
 'graph_history_points':8,'graph_history_stride':4,'path_rank':4,'path_knots':[1,2,3,4,6,8,12,16,24,32,40,48],
 'mtf_timeframes':{'M15':'15min','H1':'1h','H4':'4h','D1':'1D'},'mtf_window':32,'mtf_encoder_layers':2,
 'hidden':192,'transformer_layers':5,'attention_heads':6,'gru_layers':2,'graph_layers':2,'expert_width':512,'experts':8,'top_k':2,'graph_neighbors':6,'loader_workers':2,'prefetch_factor':2,'persistent_workers':True,'graph_index_cache':True,
  'use_bf16':True,
 'neutral_move_bps':5.,'label_vol_multiplier':.25,
 'barrier_vol_multiplier':1.,'reference_cost_bps':3.,
 'min_rows':700,'min_train_sequences':100,
 'max_train_steps_per_epoch':None,  # optional bounded research runs; no silent caps
 'direction_weight':1.,'decision_weight':1.,'quantile_weight':1.,
 'rv_weight':.5,'excursion_weight':.5,'barrier_weight':.5,'regime_weight':.25,
 'route_balance_weight':.02,'pattern_weight':.35,'jump_weight':.25,'path_weight':.35,'graph_weight':1.,
 'mtf_aux_weight':.15,
 'stage_epochs':{'A_SSL':20,'B_REGIME':10,'C_GRAPH':10,'D_FORECAST':20,'E_PATH':15,'F_EXPERTS':15,'G_DECISION':10,'H_JOINT':10},
 'stage_lr_mult':{'A_SSL':1.,'B_REGIME':.65,'C_GRAPH':.5,'D_FORECAST':.5,'E_PATH':.35,'F_EXPERTS':.35,'G_DECISION':.5,'H_JOINT':.15},
 'stage_patience':{'A_SSL':5,'B_REGIME':4,'C_GRAPH':4,'D_FORECAST':6,'E_PATH':5,'F_EXPERTS':5,'G_DECISION':4,'H_JOINT':5},
 'stage_max_val_steps':None, # None means full validation
 'research_adverse_penalty':0.20, 'utility_temperature':0.35,
 'path_nll_weight':.20,'action_q_weight':.35,'pattern_detail_weight':.25,
 'use_pcgrad_joint':True,'expert_val_batches':3,
 'validate_expert_ablation':True,'diagnostic_rolling_windows':4,
 'run_sealed_test':False,  # explicit one-time review after freeze
}
assert 3<=CFG['train_years']<=5 and CFG['recent_half_life_days']>0
assert 0<CFG['recent_uniform_floor']<1 and CFG['stress_boost']>=1
assert max(CFG['horizons'])<CFG['purge_bars']

assert set(CFG['finetune_epochs'])==set(CFG['stage_epochs']) and all(e>0 for e in CFG['finetune_epochs'].values())
assert 0<CFG['finetune_lr_multiplier']<=1 and CFG['dollar_bar_target_usd']>0
assert CFG['assets']==CFG['pool_assets']==['BTCUSD','ETHUSDT','SOLUSDT'], 'Pooled crypto asset contract'
assert CFG['target_asset']=='BTCUSD' and CFG['context_assets']==['NSXUSD']
assert CFG['path_knots'][-1] == max(CFG['horizons']) and sorted(set(CFG['path_knots'])) == CFG['path_knots']
# User switch: keep FP32 when BF16 is disabled even on L4.
USE_BF16=bool(USE_BF16 and CFG.get('use_bf16',True))

CFG['output_dir']=str(LAB_DIR/'checkpoints')
CFG['cache_dir']=str(LAB_DIR/'cache')
CFG['data_dir']=str(LAB_DIR/'input')
CFG['run_sealed_test']=False
CFG['loader_workers']=0  # synthetic benchmark uses in-memory batches; actual DataLoader assessed separately
assert 'v730_pooled_crypto_btc_finetune_checkpoints' not in CFG['output_dir']
LAB_BATCH_SIZE=8   # measured batch size; change to 16/32/64 for memory/throughput comparison
LAB_WARMUP_BATCHES=1
LAB_MEASURE_BATCHES=2
LAB_REPEAT_VALIDATION_BATCHES=1
LAB_DRY_RUN_STAGES=True
LAB_AUDIT_DRIVE=False  # set True after mounting Google Drive
assert LAB_BATCH_SIZE>0 and LAB_MEASURE_BATCHES>=1
print('Full model', CFG['hidden'],CFG['transformer_layers'],CFG['experts'],'batch=',LAB_BATCH_SIZE)


In [ ]:
# 3. Exact features + FULL model copied from V7.3.1
FEATURES=['ret1','ret3','ret12','vol12','vol48','vol_ratio','efficiency','range_norm','position','downside','skew24','kurt24','entropy24','spread_rel','spread_present','volume_log','volume_present','trend_strength_96','volatility_har_12','volatility_har_48','volatility_har_288','hurst_proxy','wavelet_band','jump_proxy','mean_reversion_z','autocorr24','taker_dollar_imbalance','taker_buy_ratio','notional_log','trade_count_log','imbalance_ewm','flow_shock','dollar_event_count_log','dollar_event_imbalance','dollar_event_activity']
H=CFG['horizons']
TF_NAMES=list(CFG['mtf_timeframes'])
class CausalDiagonalStateSpace(nn.Module):
    """Learnable stable diagonal linear recurrence using parallel causal convolution.
    Kernel k_j=(1-alpha)*alpha**j; all outputs at t use inputs <=t only.
    """
    def __init__(self,dim):
        super().__init__()
        self.decay_logits=nn.Parameter(torch.linspace(.5,5.,dim))
        self.in_projection=nn.Linear(dim,dim)
        self.out_gate=nn.Linear(dim,dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self,h):
        B,T,D=h.shape
        alpha=torch.sigmoid(self.decay_logits).clamp(.05,.999)
        lags=torch.arange(T,device=h.device,dtype=h.dtype)
        impulse=(1-alpha[:,None])*alpha[:,None].pow(lags[None,:])
        states=F.conv1d(self.in_projection(h).transpose(1,2),
                        impulse.flip(-1).unsqueeze(1),padding=T-1,groups=D)[...,:T]
        return self.norm(states.transpose(1,2)*torch.sigmoid(self.out_gate(h)))


class DynamicTemporalGraph(nn.Module):
    def __init__(self,dim,heads=6,layers=2):
        super().__init__()
        self.layers=nn.ModuleList([nn.MultiheadAttention(dim,heads,batch_first=True) for _ in range(layers)])
        self.norms=nn.ModuleList([nn.LayerNorm(dim) for _ in range(layers)])
        self.temporal=nn.GRU(dim,dim,batch_first=True)
        self.fuse=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,center,center_history,nodes,mask):
        # [B,N,T,D] per-time dynamic neighbor attention; every snapshot is <= center anchor.
        B,N,T,D=nodes.shape
        assert center_history.shape==(B,T,D), 'Graph center states must align with graph snapshots'
        center_repeat=center_history.reshape(B*T,1,D)
        z=nodes.permute(0,2,1,3).reshape(B*T,N,D)
        valid=mask.permute(0,2,1).reshape(B*T,N)
        tokens=torch.cat([center_repeat,z],dim=1)
        padding=torch.cat([torch.zeros((B*T,1),dtype=torch.bool,device=center.device),~valid],dim=1)
        for attn,norm in zip(self.layers,self.norms):
            upd,_=attn(tokens,tokens,tokens,key_padding_mask=padding,need_weights=False)
            tokens=norm(tokens+upd)
        seq=tokens[:,0].reshape(B,T,D)
        state=self.temporal(seq)[0][:,-1]
        any_neighbor=mask.any((1,2))
        fused=self.fuse(torch.cat([center,state],dim=-1))
        return torch.where(any_neighbor[:,None],fused,center)


class MultiTimeframeFusion(nn.Module):
    """Independent causal M15/H1/H4/D1 encoders plus mask-aware attention to M5.
    Historical windows end at the last CLOSED bar available at the M5 decision time.
    Missing timeframes have zero attention probability, not merely zero-filled inputs.
    """
    def __init__(self,features,hidden,heads,layers):
        super().__init__()
        self.encoders=nn.ModuleList([nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden,heads,hidden*2,dropout=.1,batch_first=True,norm_first=True),
            layers,enable_nested_tensor=False) for _ in TF_NAMES])
        self.input=nn.ModuleList([nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden)) for _ in TF_NAMES])
        self.tokens=nn.Parameter(torch.randn(len(TF_NAMES),hidden)*.02)
        self.query=nn.Linear(hidden,hidden)
        self.key=nn.Linear(hidden,hidden)
        self.value=nn.Linear(hidden,hidden)
        self.output=nn.Sequential(nn.Linear(hidden*2,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.norm=nn.LayerNorm(hidden)
        self.scale=hidden**-.5
    def forward(self,query,mtf,mtf_mask):
        B,K,W,Fdim=mtf.shape
        assert K==len(TF_NAMES) and mtf_mask.shape==(B,K,W)
        tokens=[];present=[]
        causal=torch.triu(torch.ones(W,W,device=mtf.device,dtype=torch.bool),diagonal=1)
        for j in range(K):
            good=mtf_mask[:,j,:].bool()
            # all-zero samples must still produce finite outputs; output suppressed by present mask.
            h=self.input[j](mtf[:,j])+self.tokens[j]
            h=self.encoders[j](h,mask=causal)
            idx=good.long().sum(1).sub(1).clamp(min=0)
            # Padding on left: the last candle is always W-1 if this timeframe exists.
            token=h[:,-1]
            valid=good.any(1)
            tokens.append(token);present.append(valid)
        t=torch.stack(tokens,dim=1)
        available=torch.stack(present,dim=1)
        scores=(self.query(query)[:,None,:]*self.key(t)).sum(-1)*self.scale
        scores=scores.masked_fill(~available,-1e9)
        alpha=F.softmax(scores,dim=-1)*available.float()
        alpha=alpha/alpha.sum(-1,keepdim=True).clamp(min=1e-8)
        fused=(alpha.unsqueeze(-1)*self.value(t)).sum(1)
        blended=self.output(torch.cat([query,fused],dim=-1))
        return torch.where(available.any(1,keepdim=True),blended,query),alpha


class UniversalQuantModel(nn.Module):
    def __init__(self,features,hidden=192,nexperts=8,topk=2,nh=4,
                 transformer_layers=5,attention_heads=6,gru_layers=2,graph_layers=2,expert_width=512):
        super().__init__()
        self.nh=nh;self.topk=topk;self.npath=len(CFG['path_knots']);self.rank=CFG['path_rank']
        self.input=nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden))
        self.conv=nn.Conv1d(hidden,hidden,5,padding=4,groups=hidden)
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(hidden,attention_heads,hidden*4,
            dropout=.1,batch_first=True,norm_first=True),transformer_layers,enable_nested_tensor=False)
        self.longrange=nn.GRU(hidden,hidden,num_layers=gru_layers,batch_first=True)
        self.ssm=CausalDiagonalStateSpace(hidden)
        self.graph_input=nn.Sequential(nn.Linear(features,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.graph=DynamicTemporalGraph(hidden,attention_heads,graph_layers)
        self.graph_fusion=nn.Sequential(nn.Linear(hidden*4,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.mtf=MultiTimeframeFusion(features,hidden,attention_heads,CFG['mtf_encoder_layers'])
        self.ssl_predict=nn.Linear(hidden,features)
        self.regime=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,7))
        self.router=nn.Sequential(nn.Linear(hidden+7,hidden),nn.GELU(),nn.Linear(hidden,nexperts))
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(hidden,expert_width),nn.GELU(),
                           nn.Linear(expert_width,expert_width),nn.GELU(),nn.Linear(expert_width,hidden))
                           for _ in range(nexperts)])
        # Direct connections from original market context, graph and regime; no mandatory MoE bottleneck.
        self.forecast_fusion=nn.Sequential(nn.Linear(hidden*3+7,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.direction=nn.Linear(hidden,nh*3)
        self.quantiles=nn.Linear(hidden,nh*3)
        self.rv=nn.Linear(hidden,nh)
        self.excursions=nn.Linear(hidden,nh*2)
        self.barrier=nn.Linear(hidden,nh*3)
        self.pattern=nn.Linear(hidden,nh)
        self.pattern_detail=nn.Linear(hidden,nh*3)
        self.jump=nn.Linear(hidden,nh)
        self.path=nn.Linear(hidden,nh) # legacy coarse shape, retained for continuity
        self.path_distribution=nn.Linear(hidden,self.npath*(2+self.rank))
        self.decision_quantiles=nn.Linear(hidden+nh*3,3*3)  # action utility q10/q50/q90
        self.decision=nn.Sequential(nn.Linear(hidden+nh*3+9,hidden),nn.GELU(),nn.Linear(hidden,3))
    def forward(self,x,neighbors=None,neighbor_mask=None,use_experts=True,ablated_expert=None,
                use_graph=True,mtf=None,mtf_mask=None,use_mtf=True):
        B,T,_=x.shape
        h=self.input(x)
        h=h+self.conv(h.transpose(1,2))[...,:T].transpose(1,2)
        causal=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        temporal_states=self.encoder(h,mask=causal)
        temporal=temporal_states[:,-1]
        recurrent=self.longrange(h)[0][:,-1]
        state_space=self.ssm(h)[:,-1]
        graph_repr=temporal
        if use_graph and neighbors is not None and neighbors.shape[1]>0:
            assert neighbors.ndim==4 and neighbor_mask.ndim==3
            nodes=self.graph_input(neighbors)
            gp=neighbors.shape[2];stride=CFG['graph_history_stride']
            history=temporal_states[:,-1-(gp-1)*stride::stride]
            if history.shape[1]!=gp:raise ValueError('Lookback shorter than graph history span')
            graph_repr=self.graph(temporal,history,nodes,neighbor_mask.bool())
        z0=self.graph_fusion(torch.cat([temporal,recurrent,state_space,graph_repr],-1))
        if use_mtf and mtf is not None:
            assert mtf_mask is not None
            z0,mtf_attention=self.mtf(z0,mtf,mtf_mask)
        else:
            mtf_attention=torch.zeros(B,len(TF_NAMES),device=x.device,dtype=x.dtype)
        regime=self.regime(z0)
        route=self.router(torch.cat([z0,torch.sigmoid(regime)],-1))
        top=route.topk(min(self.topk,route.shape[-1]),dim=-1)
        gate=torch.zeros_like(route).scatter(1,top.indices,F.softmax(top.values,dim=-1))
        if ablated_expert is not None:
            gate=gate.clone();gate[:,int(ablated_expert)]=0.
            gate=gate/gate.sum(1,keepdim=True).clamp(min=1e-8)
        # Sparse dispatch: only run experts selected for each sample.
        mixture=torch.zeros_like(z0)
        if use_experts:
            for j,expert in enumerate(self.experts):
                selected=torch.nonzero(gate[:,j]>0,as_tuple=True)[0]
                if selected.numel():
                    mixture=mixture.index_add(0,selected,expert(z0.index_select(0,selected))*gate[selected,j,None])
        z=z0+mixture
        ctx=z+self.forecast_fusion(torch.cat([z,z0,graph_repr,torch.sigmoid(regime)],dim=-1))
        direction=self.direction(ctx).reshape(B,self.nh,3)
        raw=self.quantiles(ctx).reshape(B,self.nh,3)
        median=raw[...,1]*ret_scale
        quantiles=torch.stack([median-F.softplus(raw[...,0])*ret_scale,median,
                               median+F.softplus(raw[...,2])*ret_scale],dim=-1)
        raw_path=self.path_distribution(ctx).reshape(B,self.npath,2+self.rank)
        path_mu=raw_path[...,0]*ret_scale
        path_diag=(F.softplus(raw_path[...,1]).clamp(max=10.)+.03)*ret_scale
        path_factors=torch.tanh(raw_path[...,2:])*ret_scale
        direction_probs=F.softmax(direction,dim=-1).reshape(B,-1)
        dq=self.decision_quantiles(torch.cat([ctx,direction_probs],dim=-1)).reshape(B,3,3)
        utility_median=dq[...,1]
        action_quantiles=torch.stack([utility_median-F.softplus(dq[...,0]),utility_median,
                                    utility_median+F.softplus(dq[...,2])],dim=-1)*ret_scale
        decision=self.decision(torch.cat([ctx,direction_probs,action_quantiles.reshape(B,-1)/ret_scale],-1))
        return {'direction':direction,'quantiles':quantiles,'rv':F.softplus(self.rv(ctx))*rv_scale,
                'excursions':self.excursions(ctx).reshape(-1,self.nh,2)*ret_scale,
                'barrier':self.barrier(ctx).reshape(-1,self.nh,3),'pattern':self.pattern(ctx),
                'pattern_detail':self.pattern_detail(ctx).reshape(-1,self.nh,3),
                'jump':self.jump(ctx),'path':self.path(ctx),
                'path_mu':path_mu,'path_diag':path_diag,'path_factors':path_factors,
                'action_quantiles':action_quantiles,'decision':decision,'regime':regime,
                'ssl_prediction':self.ssl_predict(z0),'gate':gate,
                'router_probs':F.softmax(route,dim=-1),'dispatch':(gate>0).float(),
                'mtf_attention':mtf_attention}


def path_distribution_nll(o,target):
    dist=torch.distributions.LowRankMultivariateNormal(
        loc=o['path_mu']/ret_scale,cov_factor=o['path_factors']/ret_scale,
        cov_diag=(o['path_diag']/ret_scale).square().clamp(min=1e-5))
    return (-dist.log_prob(target/ret_scale)).mean()


@torch.no_grad()
def path_forecast_quantiles(o):
    # True path distribution: cumulative correlated Gaussian increments at future knots.
    mu=o['path_mu'].cumsum(-1)
    diag_var=o['path_diag'].square().cumsum(-1)
    factor_sum=o['path_factors'].cumsum(1)
    var=diag_var+factor_sum.square().sum(-1)
    sd=var.clamp(min=1e-15).sqrt()
    return torch.stack([mu-1.2815515655*sd,mu,mu+1.2815515655*sd],dim=-1)


@torch.no_grad()
def sample_correlated_price_paths(o,starting_price,scenarios=64):
    # Draw joint log-return increments (preserves learned cross-horizon covariance).
    dist=torch.distributions.LowRankMultivariateNormal(
        loc=o['path_mu'],cov_factor=o['path_factors'],
        cov_diag=o['path_diag'].square().clamp(min=1e-14))
    increments=dist.sample((int(scenarios),)) # [scenarios, batch, path_knots]
    return float(starting_price)*increments.cumsum(-1).exp()

model=UniversalQuantModel(len(FEATURES),CFG['hidden'],CFG['experts'],CFG['top_k'],len(H),CFG['transformer_layers'],CFG['attention_heads'],CFG['gru_layers'],CFG['graph_layers'],CFG['expert_width']).to(DEVICE)
PARAMETERS=sum(p.numel() for p in model.parameters())
print('FULL model trainable parameters',PARAMETERS)
assert PARAMETERS>5_000_000, 'Unexpected smaller architecture; investigate.'


In [ ]:
# 4. Exact V7.3.1 loss/stage logic (no training loop side effects)
# These functions are copied from the production notebook source; stage training below is diagnostic only.
ret_scale=torch.tensor(0.01,device=DEVICE)
rv_scale=torch.tensor(0.01,device=DEVICE)
def pinball(pred,y):
    tau=torch.tensor([.1,.5,.9],dtype=pred.dtype,device=pred.device)
    error=y.unsqueeze(-1)-pred
    return torch.maximum(tau*error,(tau-1)*error).mean()


def cost_aware_utilities(r,m,n):
    # Research-only, future teacher; NEVER available to model during inference.
    terminal=r[:,-1]
    long=terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*(-n[:,-1]).clamp(min=0)
    short=-terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*m[:,-1].clamp(min=0)
    return torch.stack([short,torch.zeros_like(terminal),long],dim=-1)


def regime_loss(o,b):
    target=b[7];mask=b[13]
    assert target.shape[1]==7 and mask.shape==target.shape
    err=F.binary_cross_entropy_with_logits(o['regime'],target,reduction='none')
    return (err*mask).sum()/mask.sum().clamp(min=1)


def action_loss(o,b):
    util=cost_aware_utilities(b[1],b[3],b[4]).detach()
    teacher=F.softmax((util/ret_scale)/CFG['utility_temperature'],dim=-1)
    classification=-(teacher*F.log_softmax(o['decision'],dim=-1)).sum(-1).mean()
    quantile=pinball(o['action_quantiles']/ret_scale,util/ret_scale)
    return CFG['decision_weight']*(classification+CFG['action_q_weight']*quantile)


def router_balance(o):
    usage=o['dispatch'].mean(0).detach()
    probs=o['router_probs'].mean(0)
    return len(usage)*(usage*probs).sum()


def forecast_losses(o,b,include_regime=True,include_paths=True,include_route=True):
    # Mandatory future outcomes in b are only read for LOSS, never forwarded into the model.
    r,d,m,n,v,decision,pattern,jump,path=b[1],b[2],b[3],b[4],b[5],b[6],b[8],b[9],b[10]
    barrier=b[16]
    total=(CFG['direction_weight']*F.cross_entropy(o['direction'].reshape(-1,3),d.reshape(-1))+
           CFG['quantile_weight']*pinball(o['quantiles']/ret_scale,r/ret_scale)+
           CFG['rv_weight']*F.smooth_l1_loss(torch.log1p(o['rv']/rv_scale),torch.log1p(v/rv_scale))+
           CFG['excursion_weight']*.5*(F.smooth_l1_loss(o['excursions'][...,0]/ret_scale,m/ret_scale)+
             F.smooth_l1_loss(o['excursions'][...,1]/ret_scale,n/ret_scale))+
           CFG['barrier_weight']*F.cross_entropy(o['barrier'].reshape(-1,3),barrier.reshape(-1))+
           CFG['pattern_weight']*F.binary_cross_entropy_with_logits(o['pattern'],pattern)+
           CFG['pattern_detail_weight']*F.binary_cross_entropy_with_logits(o['pattern_detail'],b[14])+
           CFG['jump_weight']*F.binary_cross_entropy_with_logits(o['jump'],jump)+
           CFG['path_weight']*F.smooth_l1_loss(o['path'],path))
    if include_paths:
        total=total+CFG['path_nll_weight']*path_distribution_nll(o,b[15])
    if include_regime:
        total=total+CFG['regime_weight']*regime_loss(o,b)
    if include_route:
        total=total+CFG['route_balance_weight']*router_balance(o)
    return total


def objective(o,b):
    return forecast_losses(o,b)+action_loss(o,b)


def joint_pcgrad_backward(o,b):
    # Two-objective PCGrad on shared parameters; private-head gradients remain intact.
    predictive=forecast_losses(o,b);decision_obj=action_loss(o,b)
    params=[p for p in model.parameters() if p.requires_grad]
    a=torch.autograd.grad(predictive,params,retain_graph=True,allow_unused=True)
    g=torch.autograd.grad(decision_obj,params,allow_unused=True)
    common=[i for i,(x,y) in enumerate(zip(a,g)) if x is not None and y is not None]
    if common:
        dot=sum((a[i]*g[i]).sum() for i in common)
        aa=sum(a[i].square().sum() for i in common).clamp(min=1e-12)
        gg=sum(g[i].square().sum() for i in common).clamp(min=1e-12)
        cosine=float((dot/torch.sqrt(aa*gg)).detach())
        conflict=bool(dot.detach().item()<0)
        alpha=(dot/gg).detach() if conflict else None
        beta=(dot/aa).detach() if conflict else None
    else:
        cosine=float('nan');conflict=False;alpha=beta=None
    for i,p in enumerate(params):
        x,y=a[i],g[i]
        if x is not None and y is not None and conflict:
            p.grad=(x-alpha*y+y-beta*x).detach()
        elif x is not None and y is not None:p.grad=(x+y).detach()
        elif x is not None:p.grad=x.detach()
        elif y is not None:p.grad=y.detach()
    return {'gradient_cosine':cosine,'gradient_conflict':conflict}


def select_stage_parameters(stage):
    temporal=('input','conv','encoder','longrange','ssm','graph_fusion','mtf')
    graph=('graph_input','graph','mtf')
    forecast=('mtf','direction','quantiles','rv','excursions','barrier','pattern','pattern_detail','jump','path','forecast_fusion')
    if stage=='A_SSL':prefixes=temporal+('ssl_predict',)
    elif stage=='B_REGIME':prefixes=temporal+('regime',)
    elif stage=='C_GRAPH':prefixes=graph+('graph_fusion','forecast_fusion','direction','quantiles')
    elif stage=='D_FORECAST':prefixes=temporal+graph+('regime',)+forecast
    elif stage=='E_PATH':prefixes=temporal+graph+('forecast_fusion','path_distribution')
    elif stage=='F_EXPERTS':prefixes=temporal+graph+('regime','router','experts','path_distribution')+forecast
    elif stage=='G_DECISION':prefixes=('decision','decision_quantiles')
    elif stage=='H_JOINT':prefixes=tuple({s.split('.')[0] for s,_ in model.named_parameters()})
    else:raise ValueError(stage)
    for name,p in model.named_parameters():p.requires_grad=name.split('.')[0] in prefixes
    active=sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert active>0
    return active


def stage_step(stage,b):
    x=b[0]
    if stage=='A_SSL':
        o=model(x[:,:-1],None,None,use_experts=False,use_mtf=False)
        return F.smooth_l1_loss(o['ssl_prediction'],x[:,-1].clamp(-12,12))
    has_experts=stage in ('F_EXPERTS','G_DECISION','H_JOINT')
    graph_on=stage not in ('B_REGIME',)
    o=model(x,b[11] if graph_on else None,b[12] if graph_on else None,use_experts=has_experts,mtf=b[17],mtf_mask=b[18])
    if stage=='B_REGIME':return regime_loss(o,b)
    if stage=='C_GRAPH':
        return F.cross_entropy(o['direction'].reshape(-1,3),b[2].reshape(-1)) + pinball(o['quantiles']/ret_scale,b[1]/ret_scale)
    if stage=='D_FORECAST':return forecast_losses(o,b,include_paths=False,include_route=False)
    if stage=='E_PATH':return path_distribution_nll(o,b[15])+F.smooth_l1_loss(o['path'],b[10])
    if stage=='F_EXPERTS':return forecast_losses(o,b,include_paths=True,include_route=True)
    if stage=='G_DECISION':return action_loss(o,b)
    if stage=='H_JOINT':return objective(o,b)
    raise ValueError(stage)


def atomic_save(obj,path):
    temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)


def rng_snapshot():
    snapshot={'torch':torch.get_rng_state(),'numpy':np.random.get_state(),'python':random.getstate()}
    if torch.cuda.is_available():snapshot['cuda']=torch.cuda.get_rng_state_all()
    return snapshot


def rng_restore(snapshot):
    torch.set_rng_state(snapshot['torch']);np.random.set_state(snapshot['numpy']);random.setstate(snapshot['python'])
    if torch.cuda.is_available() and 'cuda' in snapshot:torch.cuda.set_rng_state_all(snapshot['cuda'])

STAGES=list(CFG['stage_epochs'])
assert STAGES==['A_SSL','B_REGIME','C_GRAPH','D_FORECAST','E_PATH','F_EXPERTS','G_DECISION','H_JOINT']


In [ ]:
# 5. Build strictly shape-compatible synthetic 19-element batch
# The purpose is kernel/tensor/gradient validation, NOT to claim real-data accuracy.
Fdim=len(FEATURES);nh=len(H);npath=len(CFG['path_knots']);hp=CFG['graph_history_points']
neighbor_count=3 # BTC/ETH/SOL/NSX: three neighbors per asset
def synthetic_batch(batch_size=LAB_BATCH_SIZE):
    B=batch_size;d=DEVICE
    rand=lambda *size:torch.randn(*size,device=d,dtype=torch.float32)
    x=(rand(B,CFG['lookback'],Fdim)*0.1).clamp(-2,2)
    r=(rand(B,nh)*0.005)
    ydir=torch.randint(0,3,(B,nh),device=d)
    mfe=rand(B,nh).abs()*0.003
    mae=-rand(B,nh).abs()*0.003
    rv=rand(B,nh).abs()*0.005
    decision=torch.randint(0,3,(B,),device=d)
    regime=torch.rand(B,7,device=d)
    pattern=torch.rand(B,nh,device=d)
    jump=torch.rand(B,nh,device=d)
    path=rand(B,nh)*0.005
    neighbors=rand(B,neighbor_count,hp,Fdim)*0.1
    neighbor_mask=torch.ones(B,neighbor_count,hp,device=d,dtype=torch.bool)
    regime_mask=torch.ones(B,7,device=d)
    pattern_detail=torch.rand(B,nh,3,device=d)
    future_path=rand(B,npath)*0.002
    barriers=torch.randint(0,3,(B,nh),device=d)
    mtf=rand(B,len(CFG['mtf_timeframes']),CFG['mtf_window'],Fdim)*0.1
    mtf_mask=torch.ones(B,len(CFG['mtf_timeframes']),CFG['mtf_window'],device=d,dtype=torch.bool)
    b=(x,r,ydir,mfe,mae,rv,decision,regime,pattern,jump,path,neighbors,neighbor_mask,regime_mask,pattern_detail,future_path,barriers,mtf,mtf_mask)
    assert len(b)==19 and b[0].shape[-1]==len(FEATURES)
    return b
b=synthetic_batch()
with torch.inference_mode(),amp_context():
    out=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
print('Forward check', {k:tuple(v.shape) for k,v in out.items() if isinstance(v,torch.Tensor) and v.ndim<=3})


In [ ]:
# 6. Full architecture stage smoke + synchronized GPU timing
# IMPORTANT: Pooled and BTC-finetune are phase names; these synthetic batches have NO real asset identity.
# Data routing/isolation is audited separately, not proven by this synthetic smoke test.
from time import perf_counter
if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
results=[]
def sync():
    if DEVICE.type=='cuda':torch.cuda.synchronize()
def bench_stage(phase,stage,steps,bs):
    active=select_stage_parameters(stage)
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=CFG['lr']*CFG['stage_lr_mult'][stage]*(CFG['finetune_lr_multiplier'] if phase=='btc_finetune' else 1.))
    t=[];losses=[];grad_norms=[]
    for step in range(steps+LAB_WARMUP_BATCHES):
        batch=synthetic_batch(bs)
        optimizer.zero_grad(set_to_none=True)
        sync();t0=perf_counter()
        with amp_context():
            if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
                o=model(batch[0],batch[11],batch[12],mtf=batch[17],mtf_mask=batch[18]);loss=objective(o,batch)
            else: loss=stage_step(stage,batch)
        if not torch.isfinite(loss):raise FloatingPointError(f'{phase}/{stage} loss nonfinite')
        if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
            with amp_context(): joint_pcgrad_backward(o,batch)
        else: loss.backward()
        norm=nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
        if not torch.isfinite(norm):raise FloatingPointError(f'{phase}/{stage} gradient nonfinite')
        optimizer.step();sync()
        if step>=LAB_WARMUP_BATCHES:
            t.append(perf_counter()-t0);losses.append(float(loss.detach()));grad_norms.append(float(norm))
    del optimizer;gc.collect()
    return dict(phase=phase,stage=stage,batch_size=bs,trainable=active,seconds_per_step=float(np.median(t)),samples_per_second=float(bs/np.median(t)),loss=float(np.mean(losses)),grad_norm=float(np.mean(grad_norms)),peak_gpu_memory_gb=float(torch.cuda.max_memory_allocated()/1e9) if DEVICE.type=='cuda' else None)
for phase in ['pooled','btc_finetune']:
    for stage in STAGES:
        entry=bench_stage(phase,stage,LAB_MEASURE_BATCHES,LAB_BATCH_SIZE)
        results.append(entry);print(phase,stage,'sec/step',round(entry['seconds_per_step'],3),'loss',round(entry['loss'],5))
print('STAGES PASSED:',len(results),'of 16')


In [ ]:
# 7. Checkpoint/resume roundtrip; only lab directory is touched
checkpoint=LAB_DIR/'checkpoints'/'diagnostic_roundtrip.pt'
checkpoint.parent.mkdir(exist_ok=True,parents=True)
saved={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
torch.save({'source_sha256':SOURCE_NOTEBOOK_SHA256,'model':saved,'epoch':1,'phase':'synthetic_test','stage':'H_JOINT'},checkpoint)
loaded=torch.load(checkpoint,map_location='cpu',weights_only=False)
assert loaded['source_sha256']==SOURCE_NOTEBOOK_SHA256
assert all(torch.equal(loaded['model'][k],v) for k,v in saved.items())
model.load_state_dict(loaded['model']);print('checkpoint state_dict roundtrip PASS',checkpoint)
# Check optimizer state, stage/epoch and RNG in a separate sandbox-only resume snapshot.
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-5)
optimizer.zero_grad(set_to_none=True)
test_b=synthetic_batch(2)
with amp_context():test_loss=stage_step('G_DECISION',test_b)
test_loss.backward();optimizer.step()
resume=LAB_DIR/'checkpoints'/'diagnostic_resume.pt'
state={'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
       'optimizer':optimizer.state_dict(),'stage':'G_DECISION','epoch':1,'rng':rng_snapshot()}
torch.save(state,resume)
restored=torch.load(resume,map_location='cpu',weights_only=False)
other=torch.optim.AdamW(model.parameters(),lr=1e-5)
other.load_state_dict(restored['optimizer'])
assert restored['epoch']==1 and restored['stage']=='G_DECISION'
assert len(other.state_dict()['state'])>0
assert all(torch.equal(restored['model'][k],v) for k,v in state['model'].items())
print('optimizer+model+RNG snapshot roundtrip PASS; process-restart resume still untested',resume)



In [ ]:
# 8. Optional READ-ONLY archive audit on Google Drive
# Change LAB_AUDIT_DRIVE=True in Cell 2 and run this cell with Drive mounted.
# Never downloads or extracts all data. A few ZIP records validate parser assumptions.
def audit_archives():
    checks={}; samples={}; targets={'BTCUSD':(Path(CFG['binance_archive_dir']),'BTCUSD_PERP'),
      'ETHUSDT':(Path(CFG['binance_aux_archive_base'])/'ETHUSDT','ETHUSDT'),
      'SOLUSDT':(Path(CFG['binance_aux_archive_base'])/'SOLUSDT','SOLUSDT'),
      'NSXUSD':(Path(CFG['histdata_nsx_zip_dir']),'NSXUSD')}
    for asset,(directory,symbol) in targets.items():
        files=sorted(directory.glob('*.zip')) if directory.exists() else []
        failures=[];sample=None
        if files:
            try:
                with zipfile.ZipFile(files[0]) as z:
                    bad=z.testzip() if files[0].stat().st_size<100_000_000 else None
                    if bad:failures.append('CRC:'+bad)
                    csv=[x for x in z.namelist() if x.lower().endswith('.csv')]
                    if not csv:failures.append('NO_CSV')
                    else:
                        with z.open(csv[0]) as f:
                            for _ in range(2):
                                line=f.readline().decode('utf8','replace').strip()
                                if line:sample=line[:160];break
            except Exception as exc: failures.append(repr(exc))
        checks[asset]={'directory':str(directory),'zip_count':len(files),'first_zip':files[0].name if files else None,
                       'last_zip':files[-1].name if files else None,'sample_record':sample,'errors':failures}
    return checks
if LAB_AUDIT_DRIVE:
    archive_audit=audit_archives();print(json.dumps(archive_audit,indent=2))
else:
    archive_audit=None;print('SKIPPED: set LAB_AUDIT_DRIVE=True after mounting Google Drive to inspect existing raw data.')


In [ ]:
# 9. Data-aware (optional) time projection with explicit assumptions
# Counts should be collected from the production dataset AFTER actual pre-processing.
# If counts are unknown, no full-training duration is claimed.
REAL_POOLED_TRAIN_SAMPLES=None
REAL_POOLED_VAL_SAMPLES=None
REAL_BTC_TRAIN_SAMPLES=None
REAL_BTC_VAL_SAMPLES=None
# E.g. obtain these from LazyFold(0,pool), LazyFold(1,pool), LazyFold(0,['BTCUSD']), LazyFold(1,['BTCUSD']).
# Times here are from synthetic model backward only; disk loading and validation are unmeasured.
projection=[]
for row in results:
    phase=row['phase']
    count=REAL_POOLED_TRAIN_SAMPLES if phase=='pooled' else REAL_BTC_TRAIN_SAMPLES
    if count is None:continue
    stages_epochs=CFG['stage_epochs'] if phase=='pooled' else CFG['finetune_epochs']
    train_steps=math.ceil(count/CFG['batch_size'])
    # Ratio to benchmark batch size is NOT linear for GPU kernels. Warn until measured at production batch size.
    if LAB_BATCH_SIZE!=CFG['batch_size']: continue
    projection.append({'phase':phase,'stage':row['stage'],'max_epochs':stages_epochs[row['stage']],
      'train_steps_per_epoch':train_steps,'train_hours_lower_bound':round(train_steps*stages_epochs[row['stage']]*row['seconds_per_step']/3600,2)})
print('Projections',projection if projection else 'UNAVAILABLE: require actual sample counts and measurement at production batch size; no 5y estimate fabricated.')


In [ ]:
# 10. Final isolated report. Includes precisely what was and was not tested.
report={'created_utc':datetime.now(timezone.utc).isoformat(),'source_notebook_sha256':SOURCE_NOTEBOOK_SHA256,
        'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
        'bf16':USE_BF16,'parameter_count':PARAMETERS,'benchmark_batch_size':LAB_BATCH_SIZE,
        'production_batch_size':CFG['batch_size'],'measured_train_steps_per_stage':LAB_MEASURE_BATCHES,
        'stages':results,'archive_audit':archive_audit,'time_projection':projection,
        'checks':{'all_16_synthetic_stages_passed':len(results)==16,'isolated_checkpoint_model_state_roundtrip':True,'optimizer_rng_snapshot_roundtrip':True},
        'not_proven':['true historical Binance/HistData data ingestion and completeness',
                      'historical data split / leakage on actual records','production Dataset/DataLoader performance',
                      'optimizer/RNG resume across process restart','true liquidation labels','model predictive edge',
                      'full 140-epoch runtime estimate unless same batch measured and real counts supplied'] }
report_path=LAB_DIR/'v731_isolated_l4_diagnostic.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str))
print('REPORT:',report_path)
print('Safe: production notebook and production checkpoints were not written.')
